### Training a neural network for image classification

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms

# Define the neural network architecture
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.dropout1 = nn.Dropout2d(0.25)
        self.dropout2 = nn.Dropout2d(0.5)
        self.fc1 = nn.Linear(64 * 14 * 14, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = nn.functional.relu(self.conv1(x))
        x = nn.functional.relu(self.conv2(x))
        x = nn.functional.max_pool2d(self.dropout1(x), 2)
        x = torch.flatten(x, 1)
        x = nn.functional.relu(self.fc1(self.dropout2(x)))
        x = self.fc2(x)
        return nn.functional.log_softmax(x, dim=1)

# Set the device to GPU if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f'Device: {device}')
print(f'Torch CUDA available: {torch.cuda.is_available()}')

# Define stage processing data
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# Load the MNIST dataset
train_dataset = datasets.MNIST('./data', train=True, download=True,
    transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)

# Create data loaders
batch_size = 64
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size,
    shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size,
    shuffle=True)

# Initialize the model and optimizer
model = Net().to(device)
optimizer = optim.Adam(model.parameters())

# Compile the model using the Torch 2.0 optimizer
model = torch.compile(model)

# Set up the training loop
model.train()
for batch_idx, (data, target) in enumerate(train_loader):
    data, target = data.to(device), target.to(device)
    optimizer.zero_grad()
    output = model(data)
    loss = nn.functional.nll_loss(output, target)
    loss.backward()
    optimizer.step()
    
# Adjust the model for evaluation
model.eval()
test_loss = 0
correct = 0

with torch.no_grad():
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        # Knowledge distillation: calculate the loss for the test set
        test_loss += nn.functional.nll_loss(
            output, target, reduction='sum'
        ).item()
        # Add the number of correct predictions to the total count
        pred = output.argmax(dim=1, keepdim=True)
        correct += pred.eq(target.view_as(pred)).sum().item()
        
test_loss /= len(test_loader.dataset)

Device: cpu
Torch CUDA available: False


/home/max/projects/machine-learning/22_nn_with_unstructured_data/.venv/lib/python3.13/site-packages/torch/_dynamo/utils.py:4231: UserWarning: dropout2d: Received a 2-D input to dropout2d, which is deprecated and will result in an error in a future release. To retain the behavior and silence this warning, please use dropout instead. Note that dropout2d exists to provide channel-wise dropout on inputs with 2 spatial dimensions, a channel dimension, and an optional batch dimension (i.e. 3D or 4D inputs).
  return node.target(*args, **kwargs)  # type: ignore[operator]
/home/max/projects/machine-learning/22_nn_with_unstructured_data/.venv/lib/python3.13/site-packages/torch/_dynamo/utils.py:4231: UserWarning: dropout2d: Received a 2-D input to dropout2d, which is deprecated and will result in an error in a future release. To retain the behavior and silence this warning, please use dropout instead. Note that dropout2d exists to provide channel-wise dropout on inputs with 2 spatial dimensions,

### Training a neural network for text classification

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Load the 20 Newsgroups dataset
cats = ['alt.atheism', 'sci.space']
newsgroups_data = fetch_20newsgroups(subset='all', shuffle=True,
    random_state=42, categories=cats)

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(newsgroups_data.data,
    newsgroups_data.target, test_size=0.2, random_state=42)

# Create vectorizer text dataset use method of bag-of-words
vectorizer = CountVectorizer(stop_words='english')
X_train = vectorizer.fit_transform(X_train).toarray()
X_test = vectorizer.transform(X_test).toarray()

# Convert the data to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.long)

# Define the model
class TextClassifier(nn.Module):
    def __init__(self, num_classes):
        super(TextClassifier, self).__init__()
        self.fc1 = nn.Linear(X_train.shape[1], 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = nn.functional.relu(self.fc1(x))
        return self.fc2(x)

# Init the model, loss function, and optimizer
model = TextClassifier(num_classes=len(cats))
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Compile the model using the optimizer in torch 2.0
model = torch.compile(model)

# Train the model
num_epochs = 1
batch_size = 10

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    num_batches = 0

    for start_idx in range(0, len(X_train), batch_size):
        # Prepare the batch data
        end_idx = start_idx + batch_size
        inputs = X_train[start_idx:end_idx]
        targets = y_train[start_idx:end_idx]

        # Set the gradients to zero before backpropagation
        optimizer.zero_grad()

        # Pass the data through the model and calculate the loss
        outputs = model(inputs)
        loss = loss_function(outputs, targets)

        # Backpropagate the error and update the parameters
        loss.backward()
        optimizer.step()

        # Update the total loss for the epoch
        total_loss += loss.item()
        num_batches += 1

    # Calculate the test accuracy after each epoch
    model.eval()
    with torch.no_grad():
        test_outputs = model(X_test)
        test_predictions = torch.argmax(test_outputs, dim=1)

    test_accuracy = accuracy_score(
        y_test.cpu().numpy(),
        test_predictions.cpu().numpy()
    )

    # Print the epoch number, average loss, and test accuracy
    print(
        f"Epoch: {epoch + 1}, Loss: {total_loss / num_batches:.4f}, "
        f"Test Accuracy: {test_accuracy:.4f}"
    )

Epoch: 1, Loss: 0.1644, Test Accuracy: 0.9916


### Fine-tuning a pre-trained model for image classification (Time 16 min in CPU 8 cores)

In [2]:
import numpy as np
import torch
from sklearn.metrics import accuracy_score
from torchvision.transforms import Compose, Normalize, Resize, ToTensor
from transformers import (
    DefaultDataCollator,
    Trainer,
    TrainingArguments,
    ViTForImageClassification,
    ViTImageProcessor,
 )
from datasets import load_dataset

# Load the Fashion MNIST dataset from the Hugging Face Hub
dataset = load_dataset("zalando-datasets/fashion_mnist")

# Use a larger, shuffled subset for better accuracy
dataset["train"] = dataset["train"].shuffle(seed=42).select(range(4096))
dataset["test"] = dataset["test"].shuffle(seed=42).select(range(512))

# Load the image processor and labels for the pretrained ViT model
image_processor = ViTImageProcessor.from_pretrained(
    "google/vit-base-patch16-224-in21k"
 )
labels = dataset["train"].features["label"].names

model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224-in21k",
    num_labels=len(labels),
    id2label={str(i): label for i, label in enumerate(labels)},
    label2id={label: str(i) for i, label in enumerate(labels)},
    ignore_mismatched_sizes=True,
 )

# Freeze the pretrained ViT and train only the new classification head
for parameter in model.vit.parameters():
    parameter.requires_grad = False

# Convert 28x28 Fashion MNIST images to the 224x224 RGB format expected by ViT
normalize = Normalize(
    mean=image_processor.image_mean,
    std=image_processor.image_std,
 )
image_transform = Compose([
    Resize((224, 224)),
    ToTensor(),
    normalize,
 ])

def transform_images(examples):
    examples["pixel_values"] = [
        image_transform(image.convert("RGB"))
        for image in examples["image"]
    ]
    del examples["image"]
    return examples

def compute_metrics(eval_prediction):
    predictions = np.argmax(eval_prediction.predictions, axis=1)
    return {
        "accuracy": accuracy_score(eval_prediction.label_ids, predictions)
    }

dataset = dataset.with_transform(transform_images)
data_collator = DefaultDataCollator()

training_args = TrainingArguments(
    output_dir="fashion_mnist_model",
    remove_unused_columns=False,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=1,
    learning_rate=0.0005,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    per_device_eval_batch_size=16,
    max_steps=500,
    warmup_steps=40,
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    push_to_hub=False,
 )

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=image_processor,
 )

print("Training the classification head on 4,096 images for 500 steps")
train_results = trainer.train()
trainer.save_model()
trainer.log_metrics("train", train_results.metrics)
trainer.save_metrics("train", train_results.metrics)
trainer.save_state()

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training the classification head on 4,096 images for 500 steps


/home/max/projects/machine-learning/22_nn_with_unstructured_data/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss,Accuracy
50,2.127300,2.028652,0.578125
100,1.772600,1.665008,0.710938
150,1.483600,1.407906,0.775391
200,1.281300,1.249014,0.771484
250,1.154900,1.122808,0.785156
300,1.054900,1.048522,0.789062
350,1.059900,0.993739,0.792969
400,0.965700,0.960423,0.798828
450,0.946800,0.940486,0.802734
500,0.917400,0.934081,0.802734


/home/max/projects/machine-learning/22_nn_with_unstructured_data/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/home/max/projects/machine-learning/22_nn_with_unstructured_data/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/home/max/projects/machine-learning/22_nn_with_unstructured_data/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/home/max/projects/machine-learning/22_nn_with_unstructured_data/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'p

***** train metrics *****
  epoch                    =      0.9766
  total_flos               = 288700855GF
  train_loss               =      1.2953
  train_runtime            =  0:16:26.32
  train_samples_per_second =       4.055
  train_steps_per_second   =       0.507


### Fine-tuning a pre-trained model for text classification (Time 9 min in CPU 8 cores)

In [1]:
import os

import evaluate
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

# Keep CPU execution responsive without consuming all system resources.
torch.set_num_threads(min(8, os.cpu_count() or 1))

# Load small, shuffled splits of IMDB suitable for local CPU fine-tuning.
imdb = load_dataset("imdb")
imdb["train"] = imdb["train"].shuffle(seed=42).select(range(4_000))
imdb["test"] = imdb["test"].shuffle(seed=42).select(range(1_000))

# Tokenize to a practical maximum length; padding happens dynamically per batch.
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def tokenize_batch(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

tokenized_imdb = imdb.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"],
)

# Define metric for evaluation.
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

# Create dictionaries to map between text labels and integer labels.
id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

# Load pretrained DistilBERT model for sequence classification with 2 labels.
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

# Configure a memory-conscious CPU training run.
training_args = TrainingArguments(
    output_dir="my_awesome_model",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=25,
    dataloader_num_workers=4,
    report_to="none",
    use_cpu=True,
)

# Adjust the model for training using the Trainer API.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_imdb["train"],
    eval_dataset=tokenized_imdb["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Train, save the model, and record training and evaluation metrics.
train_results = trainer.train()
trainer.save_model()
trainer.log_metrics("train", train_results.metrics)
trainer.save_metrics("train", train_results.metrics)

eval_results = trainer.evaluate()
trainer.log_metrics("eval", eval_results)
trainer.save_metrics("eval", eval_results)

/home/max/projects/machine-learning/22_nn_with_unstructured_data/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Map: 100%|██████████| 50000/50000 [00:05<00:00, 9762.37 examples/s] 
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.405000,0.369043,0.843000


***** train metrics *****
  epoch                    =        1.0
  total_flos               =   123369GF
  train_loss               =     0.4406
  train_runtime            = 0:08:23.79
  train_samples_per_second =       7.94
  train_steps_per_second   =      0.496


***** eval metrics *****
  epoch                   =        1.0
  eval_accuracy           =      0.843
  eval_loss               =      0.369
  eval_runtime            = 0:00:26.23
  eval_samples_per_second =     38.123
  eval_steps_per_second   =      4.765
